In [10]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
plt.style.use('ggplot')  # Use ggplot style
sns.set_palette('muted')  # Set Seaborn color palette

# Load dataset
df = pd.read_csv('credit_risk_dataset.csv')

# 1. Check data quality
print("=== Dataset Overview ===")
print(df.info())
print("\n=== Missing Values ===")
print(df.isnull().sum())
print("\n=== First 5 Rows ===")
print(df.head())

# 2. Check outliers
print("\n=== Descriptive Statistics ===")
print(df.describe())

# Define outlier detection rules
# Loan amount: Below 1000 or above 500000 EUR
# Income: Below 5000 or above 500000 EUR
# Debt-to-income ratio: Below 0 or above 1
# Credit score: Ensure within 300-850
outliers = {
    'loan_amount': ((df['loan_amount'] < 1000) | (df['loan_amount'] > 500000)).sum(),
    'income': ((df['income'] < 5000) | (df['income'] > 500000)).sum(),
    'debt_to_income': ((df['debt_to_income'] < 0) | (df['debt_to_income'] > 1)).sum(),
    'credit_score': ((df['credit_score'] < 300) | (df['credit_score'] > 850)).sum(),
}
print("\n=== Outlier Statistics ===")
for key, value in outliers.items():
    print(f"{key}: {value} records")

# 3. Handle outliers
# Loan amount: Clip to reasonable range
df['loan_amount'] = df['loan_amount'].clip(lower=1000, upper=500000)
# Income: Clip to reasonable range
df['income'] = df['income'].clip(lower=5000, upper=500000)
# Debt-to-income ratio: Clip to 0-1
df['debt_to_income'] = df['debt_to_income'].clip(lower=0, upper=1)
# Credit score: Already controlled in data generation (300-850)

# 4. Recheck descriptive statistics
print("\n=== Descriptive Statistics After Cleaning ===")
print(df.describe())

# 5. Exploratory Data Analysis - Visualization
# Create output directory
import os
if not os.path.exists('plots'):
    os.makedirs('plots')

# (1) Loan Amount Distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['loan_amount'], bins=50, kde=True)
plt.title('Loan Amount Distribution')
plt.xlabel('Loan Amount (EUR)')
plt.ylabel('Frequency')
plt.tight_layout()  # Adjust layout to prevent label cutoff
plt.savefig('plots/loan_amount_distribution.png')
plt.close()

# (2) Credit Score Distribution
plt.figure(figsize=(10, 6))
sns.histplot(df['credit_score'], bins=50, kde=True)
plt.title('Credit Score Distribution')
plt.xlabel('Credit Score')
plt.ylabel('Frequency')
plt.tight_layout()  # Adjust layout to prevent label cutoff
plt.savefig('plots/credit_score_distribution.png')
plt.close()

# (3) Default Status vs. Credit Score
plt.figure(figsize=(10, 6))
sns.boxplot(x='default_status', y='credit_score', data=df)
plt.title('Default Status vs. Credit Score')
plt.xlabel('Default Status (0=No Default, 1=Default)')
plt.ylabel('Credit Score')
plt.tight_layout()  # Adjust layout to prevent label cutoff
plt.savefig('plots/default_vs_credit_score.png')
plt.close()

# (4) Employment Status Distribution
plt.figure(figsize=(10, 6))
sns.countplot(x='employment_status', data=df)
plt.title('Employment Status Distribution')
plt.xlabel('Employment Status')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()  # Adjust layout to prevent label cutoff
plt.savefig('plots/employment_status_distribution.png')
plt.close()

# (5) Correlation Heatmap
plt.figure(figsize=(12, 10))  # Increase figure size for more space
# Shorten labels for clarity while keeping meaning
numeric_cols = ['loan_amount', 'loan_term', 'credit_score', 'income', 'debt_to_income', 'default_probability']
short_labels = ['Loan Amt', 'Loan Term', 'Credit Score', 'Income', 'Debt/Income', 'Default Prob']
corr = df[numeric_cols].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', vmin=-1, vmax=1, 
            xticklabels=short_labels, yticklabels=short_labels)
plt.title('Correlation Heatmap of Numeric Variables')
plt.tick_params(axis='both', which='major', labelsize=10)  # Adjust font size of labels
plt.tight_layout()  # Adjust layout to prevent label cutoff
plt.savefig('plots/correlation_heatmap.png')
plt.close()

# 6. Descriptive Statistics - Grouped Analysis
print("\n=== Statistics by Default Status ===")
group_stats = df.groupby('default_status')[['loan_amount', 'credit_score', 'income', 'debt_to_income']].mean().round(2)
print(group_stats)

# Save cleaned dataset
df.to_csv('credit_risk_dataset_cleaned.csv', index=False)
print("\nCleaned dataset saved as 'credit_risk_dataset_cleaned.csv'")

=== Dataset Overview ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   customer_id          20000 non-null  int64  
 1   loan_amount          20000 non-null  float64
 2   loan_term            20000 non-null  int64  
 3   credit_score         20000 non-null  float64
 4   income               20000 non-null  float64
 5   employment_status    20000 non-null  object 
 6   debt_to_income       20000 non-null  float64
 7   default_probability  20000 non-null  float64
 8   default_status       20000 non-null  int64  
dtypes: float64(5), int64(3), object(1)
memory usage: 1.4+ MB
None

=== Missing Values ===
customer_id            0
loan_amount            0
loan_term              0
credit_score           0
income                 0
employment_status      0
debt_to_income         0
default_probability    0
default_status         0
d